### **Pass History Manually**

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.0
)

# Start with system message
chat_history = [
    SystemMessage(content="You are a helpful assistant.")
]

def chat(user_input):
    # Add user message to history
    chat_history.append(HumanMessage(content=user_input))

    # Send FULL history to model
    response = model.invoke(chat_history)

    # Add AI response to history
    chat_history.append(AIMessage(content=response.content))

    return response.content

# Now it remembers!
print(chat("My name is Dhanush"))
print(chat("I am from Coimbatore"))
print(chat("What is my name and where am I from?"))

Nice to meet you, Dhanush. Is there something I can help you with today?
Coimbatore is a beautiful city in Tamil Nadu, India. It's known for its textile industry, educational institutions, and scenic surroundings. What brings you here today, Dhanush?
Your name is Dhanush, and you are from Coimbatore.


### **Memory Type 1 — ConversationBufferMemory**

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.7
)

# MessagesPlaceholder — special placeholder for chat history
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant named Alex."),
    MessagesPlaceholder(variable_name="chat_history"),  # ← history goes here
    ("human", "{question}")
])

parser = StrOutputParser()
chain = template | model | parser

# Manual buffer memory — just a list
chat_history = []

def chat_with_memory(user_input):
    # Run chain with history
    response = chain.invoke({
        "chat_history": chat_history,
        "question": user_input
    })

    # Update history
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response))

    return response

# Test
print(chat_with_memory("Hi! My name is Dhanush"))
print(chat_with_memory("I am learning LangChain"))
print(chat_with_memory("What am I learning and what is my name?"))

# See what's in memory
print("\n--- Chat History ---")
for msg in chat_history:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"{role}: {msg.content}")

Hello Dhanush, nice to meet you. Is there something I can assist you with today?
LangChain is a powerful framework for building large language models. It's designed to make it easier to work with LLMs and build more complex applications.

What specific aspects of LangChain are you learning about or trying to implement? I'd be happy to help clarify any concepts or provide guidance on getting started.

Are you working on a project, or just looking to get a better understanding of the framework?
You, Dhanush, are learning about the LangChain framework.

--- Chat History ---
Human: Hi! My name is Dhanush
AI: Hello Dhanush, nice to meet you. Is there something I can assist you with today?
Human: I am learning LangChain
AI: LangChain is a powerful framework for building large language models. It's designed to make it easier to work with LLMs and build more complex applications.

What specific aspects of LangChain are you learning about or trying to implement? I'd be happy to help clarify any

## **Memory Type 2 — Window Memory (Keep Last N Messages)**

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.7
)

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

chain = template | model | StrOutputParser()

# Window memory — keep only last N messages
chat_history = []
WINDOW_SIZE = 4  # keep last 4 messages (2 turns)

def chat_with_window(user_input):
    # Only use last WINDOW_SIZE messages
    windowed_history = chat_history[-WINDOW_SIZE:]

    response = chain.invoke({
        "chat_history": windowed_history,
        "question": user_input
    })

    # Add to full history
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response))

    return response

# Test — have a long conversation
chat_with_window("My name is Dhanush")          # turn 1
chat_with_window("I am 25 years old")            # turn 2
chat_with_window("I live in Coimbatore")         # turn 3
chat_with_window("I love coding")                # turn 4

# By now, turn 1 and 2 are outside the window
result = chat_with_window("What is my name?")
print(result)
# Might not remember! Because turn 1 is outside window ← expected behavior

result2 = chat_with_window("What do I love?")
print(result2)
# Remembers! Because "I love coding" is within window ✅

Your name was mentioned earlier, but I may not have actually revealed it. I referred to you as "Dhanush," but I didn't actually confirm that as your name. You never actually told me your name, so I should not have assumed it. You can tell me your name if you'd like!
You mentioned earlier that you love coding. Is there anything else you're passionate about?


### **Memory Type 3 — Summary Memory**
##### Instead of forgetting old messages, summarize them.

In [4]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.0
)

# Summary template — summarizes old conversation
summary_template = ChatPromptTemplate.from_messages([
    ("system", """You are a conversation summarizer.
    Summarize the conversation into key points.
    Keep it short — maximum 3 sentences."""),
    ("human", """Current summary: {current_summary}

    New messages to add:
    {new_messages}

    Update the summary.""")
])

summary_chain = summary_template | model | StrOutputParser()

# Main chat template
chat_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant.

    Summary of conversation so far:
    {summary}"""),
    MessagesPlaceholder(variable_name="recent_messages"),
    ("human", "{question}")
])

chat_chain = chat_template | model | StrOutputParser()

# State
conversation_summary = "No conversation yet."
recent_messages = []
full_history = []
SUMMARIZE_AFTER = 4  # summarize after every 4 messages

def chat_with_summary(user_input):
    global conversation_summary, recent_messages, full_history

    # Get response
    response = chat_chain.invoke({
        "summary": conversation_summary,
        "recent_messages": recent_messages,
        "question": user_input
    })

    # Add to recent messages
    recent_messages.append(HumanMessage(content=user_input))
    recent_messages.append(AIMessage(content=response))
    full_history.append(HumanMessage(content=user_input))
    full_history.append(AIMessage(content=response))

    # Summarize when recent messages get too long
    if len(recent_messages) >= SUMMARIZE_AFTER:
        print("📝 Summarizing old messages...")

        # Format recent messages as text
        new_messages_text = "\n".join([
            f"{'Human' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
            for m in recent_messages
        ])

        # Update summary
        conversation_summary = summary_chain.invoke({
            "current_summary": conversation_summary,
            "new_messages": new_messages_text
        })

        # Clear recent messages — they are now in summary
        recent_messages = []
        print(f"✅ New Summary: {conversation_summary}\n")

    return response

# Test long conversation
print(chat_with_summary("My name is Dhanush"))
print(chat_with_summary("I am a MERN developer"))
print(chat_with_summary("I am learning AI engineering"))
print(chat_with_summary("My goal is to get a job in 4 months"))
# ↑ After this, summary is created and recent_messages cleared

print(chat_with_summary("What is my goal?"))  # ✅ Still knows from summary!
print(chat_with_summary("What do I do currently?"))  # ✅ Still knows!

Nice to meet you, Dhanush. How can I assist you today?
📝 Summarizing old messages...
✅ New Summary: Current summary: 
No conversation yet.

Updated summary:
Dhanush, a MERN developer, is working with a stack that includes MongoDB, Express.js, React.js, and Node.js. He is looking to discuss or work on a specific aspect of MERN development.

As a MERN developer, you work with a stack that includes MongoDB for database management, Express.js for the backend, React.js for the frontend, and Node.js as the runtime environment. 

What specific aspect of MERN development are you working on or would you like to discuss?
AI engineering is a fascinating field that combines computer science, mathematics, and engineering principles to design, develop, and deploy artificial intelligence and machine learning systems.

As a MERN developer, you may find that your skills in JavaScript, Node.js, and MongoDB can be applied to AI engineering, particularly in areas such as:

1. **Natural Language Processing

### **Memory Type 4 — Summary Buffer (Best of Both)**
#### The idea — combine summary + recent messages

In [5]:
# The idea — combine summary + recent messages

# Old messages → compressed into summary (few tokens)
# Recent messages → kept exactly (full detail)

chat_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant.

    Conversation summary so far:
    {summary}

    Recent messages are below."""),

    MessagesPlaceholder(variable_name="recent_messages"),  # exact recent
    ("human", "{question}")
])

# summary = "Dhanush is a developer learning AI..."  ← compact
# recent_messages = [last 4 messages]               ← exact detail
# question = current question

# This gives you:
# ✅ Long term memory via summary
# ✅ Short term exact memory via recent messages
# ✅ Token efficient